In [3]:
# create the dataset
import pandas as pd
import numpy as np

np.random.seed(42)

n = 5000

data = pd.DataFrame({
    "CustomerID": range(1, n+1),
    "TenureMonths": np.random.randint(1, 60, n),
    "MonthlySpend": np.random.randint(20, 200, n),
    "SupportTickets": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(["Monthly","Annual","Two-Year"], n),
    "Churn": np.random.choice(["Yes","No"], n, p=[0.25,0.75])
})

data["TotalSpend"] = data["TenureMonths"] * data["MonthlySpend"]

data.to_csv("customer_churn.csv", index=False)

print("Dataset created successfully")

Dataset created successfully


In [12]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Iamakshay@123",
    database="churn_analysis"
)

query = "SELECT * FROM customer_churn"
df = pd.read_sql(query, conn)

print(df.head())

   CustomerID  TenureMonths  MonthlySpend  SupportTickets ContractType Churn  \
0           1            39            78               7      Monthly    No   
1           2            52           151               4      Monthly    No   
2           3            29           123               0      Monthly   Yes   
3           4            15           176               7      Monthly    No   
4           5            43           176               8      Monthly   Yes   

   TotalSpend  
0        3042  
1        7852  
2        3567  
3        2640  
4        7568  


In [15]:
df.dtypes

CustomerID         int64
TenureMonths       int64
MonthlySpend       int64
SupportTickets     int64
ContractType      object
Churn              int64
TotalSpend         int64
dtype: object

In [16]:
df["Churn"].unique()

array([0, 1], dtype=int64)

In [17]:
features = ["TenureMonths", "MonthlySpend", "SupportTickets"]
X = df[features]
y = df["Churn"]

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

model = LogisticRegression()
model.fit(X_train,y_train)

LogisticRegression()

In [20]:
df['churn_probability'] = model.predict_proba(X)[:,1]
print(df.head())

   CustomerID  TenureMonths  MonthlySpend  SupportTickets ContractType  Churn  \
0           1            39            78               7      Monthly      0   
1           2            52           151               4      Monthly      0   
2           3            29           123               0      Monthly      1   
3           4            15           176               7      Monthly      0   
4           5            43           176               8      Monthly      1   

   TotalSpend  churn_probability  
0        3042           0.242411  
1        7852           0.254708  
2        3567           0.269758  
3        2640           0.268607  
4        7568           0.253683  


In [21]:
def risk_level(p):
    if p > 0.7:
        return "High Risk"
    elif p > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df['Churn_Risk'] = df['churn_probability'].apply(risk_level)
print(df.head())

   CustomerID  TenureMonths  MonthlySpend  SupportTickets ContractType  Churn  \
0           1            39            78               7      Monthly      0   
1           2            52           151               4      Monthly      0   
2           3            29           123               0      Monthly      1   
3           4            15           176               7      Monthly      0   
4           5            43           176               8      Monthly      1   

   TotalSpend  churn_probability Churn_Risk  
0        3042           0.242411   Low Risk  
1        7852           0.254708   Low Risk  
2        3567           0.269758   Low Risk  
3        2640           0.268607   Low Risk  
4        7568           0.253683   Low Risk  
